# Feature Engineering v2 — Enriched Feature Table for Modelling

**Companion to:** `cleaning_v2.ipynb` and `val_split_v2.ipynb`. **Successor to:** *(none — this is new in v2)*.

### What this notebook does

Reads the leakage-free cleaned data plus the daytime-matched validation split, then engineers the lag, rolling-window, boundary, OOF target-encoded, and saturation features required by the modelling stage. The result is **two parquets** that every downstream model notebook consumes directly:

- `processed/train_features.parquet` — 77 299 rows, every feature plus the target.
- `processed/test_features.parquet` — 41 778 rows, every feature except the target.

It also emits:

- `processed/cv_folds.parquet` — five contiguous **time-block** CV folds covering all 105 train `t_abs` slots, so every train row has a clean out-of-fold (OOF) prediction in the model-training stage.
- `processed/saturating_geohashes.json` — the geohashes that ever observed `demand ≥ 0.99` in train (the ~23 Highway cells where the saturation residual head will fire).
- `processed/feature_meta_v2.json` — the contract for downstream notebooks: which columns are features, which are categorical (LGBM `categorical_feature=` hint), which are OOF-target-encoded (leakage-sensitive), which is the sample-weight column.

### Why the OOF / CV design matters

A naïve target encoding of `geohash → mean(demand)` would leak the row's own demand into its feature. We avoid this by:

1. Partitioning the train rows into 5 **contiguous time-blocks** (each ≈ 21 quarter-hours of day 48).
2. For every row in block *k*, computing its target encoding from rows in blocks ≠ *k* only.
3. For test rows, using the encoder fit on all 5 blocks (full train).

Time-blocking (vs random K-fold) keeps the OOF roughly causal: within each block, the encoder is built on data from *other* time windows, not future data inside the same window. The forecasting setup is approximated correctly.

### Why we **don't** rebuild any of cleaning_v2's work

`cleaning_v2.ipynb` already produced the leakage-free static features (modal road type, train-only imputed Temperature, etc.). We carry those forward verbatim and **only add** the temporal/lag/rolling/encoded features here.


## 1. Setup

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE = r"."
PROCESSED = os.path.join(BASE, "processed")
print('Reading cleaning_v2 outputs from:', PROCESSED)

Reading cleaning_v2 outputs from: .\processed


## 2. Load cleaning_v2 outputs

We keep `train_clean_v2.csv` and `test_clean_v2.csv` as our single source of truth for the static features. **Nothing about RoadType_modal, Temperature, or Weather changes in this notebook** — those are handled correctly in cleaning_v2. We only **add** new columns.

In [2]:
train = pd.read_csv(os.path.join(PROCESSED, 'train_clean_v2.csv'))
test  = pd.read_csv(os.path.join(PROCESSED, 'test_clean_v2.csv'))
print('train_clean_v2:', train.shape)
print('test_clean_v2:', test.shape)
train.head(3)

train_clean_v2: (77299, 46)
test_clean_v2: (41778, 44)


,Index,day,timestamp,hour,minute,qhour_of_day,t_abs,hour_sin,hour_cos,qhour_sin,qhour_cos,tod_bucket,tod_bucket_code,geohash,geohash_code,gh_prefix4,gh_prefix4_code,gh_prefix5,gh_prefix5_code,lat,lon,gh_train_count,pfx5_train_count,RoadType,RoadType_code,RoadType_was_missing,NumberofLanes,NumberofLanes_ord,LargeVehicles,LargeVehicles_code,Landmarks,Landmarks_code,RoadType_modal,RoadType_modal_code,NumberofLanes_modal,LargeVehicles_modal,LargeVehicles_modal_code,Landmarks_modal,Landmarks_modal_code,Weather,Weather_code,Weather_was_missing,Temperature,Temperature_was_missing,demand,demand_log1p
0,0,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp02z1,4,qp02,0,qp02z,1,-5.484924,90.664673,33,1389,Residential,1,1,1,1,Not Allowed,1,No,0,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,1,16.861615,1,0.048804,0.047651
1,1,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp02zt,25,qp02,0,qp02z,1,-5.462952,90.686646,89,1389,Residential,1,0,3,3,Allowed,0,Yes,1,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,0,31.104565,0,0.118507,0.111995
2,2,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp08bj,370,qp08,3,qp08b,16,-5.462952,90.708618,67,842,Residential,1,0,1,1,Not Allowed,1,No,0,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,0,25.919267,0,0.027132,0.026770


## 3. Combine train + test on a single timeline

Many of the features we build (lags, rolling means, boundary stats) are computed **per geohash across time**, where time extends from the start of day 48 through the end of the test horizon. So we have to combine train and test into one frame, keep an `is_train` flag, and use the cleanup-friendly `Index` column as a stable `row_id`.

In [3]:
train['is_train'] = 1
test['is_train']  = 0
test['demand']        = np.nan
test['demand_log1p']  = np.nan

train = train.rename(columns={'Index': 'row_id'})
test  = test.rename(columns={'Index': 'row_id'})

df = pd.concat([train, test], ignore_index=True, sort=False)
print('Combined:', df.shape, '  rows = train(77299) + test(41778)')
df[['row_id','geohash','day','t_abs','is_train','demand']].head(4)

Combined: (119077, 47)   rows = train(77299) + test(41778)


,row_id,geohash,day,t_abs,is_train,demand
0,0,qp02z1,48,4608,1,0.048804
1,1,qp02zt,48,4608,1,0.118507
2,2,qp08bj,48,4608,1,0.027132
3,3,qp08gt,48,4608,1,0.003272


## 4. Reindex onto the regular 15-minute grid

The audit found that **1081 / 1249 geohashes have gaps in their 15-minute grid** (some quarter-hours are simply missing from the observations). Naively computing `lag_1 = demand.shift(1)` per geohash would silently mix non-adjacent timesteps — e.g. a row at qhour 30 would get its "lag-1" from qhour 27 if qhours 28 and 29 are absent.

The fix is to **reindex every geohash onto the full 152-slot integer grid `t_abs ∈ [4608, 4759]`** (96 slots of day 48 + 56 slots up to test end), then compute lags on the dense grid. After feature computation we drop the inserted (synthetic) rows and keep only the original observations.

We also record a `was_observed` indicator so the model knows when a lag value came from a real observation vs an imputed (NaN) cell.

In [4]:
T_ABS_TRAIN_MAX = 4712
T_ABS_LO = 4608
T_ABS_HI = 4759  # day 49 qhour 55 = test end
ALL_T_ABS = np.arange(T_ABS_LO, T_ABS_HI + 1)
print(f'Dense grid: t_abs {T_ABS_LO}..{T_ABS_HI}  ({len(ALL_T_ABS)} slots = 96 + 56)')

geohashes = df['geohash'].unique()
print(f'Unique geohashes (train ∪ test): {len(geohashes)}')

# Cartesian product (geohash × t_abs)
gh_idx = pd.MultiIndex.from_product([geohashes, ALL_T_ABS], names=['geohash', 't_abs'])
grid = pd.DataFrame(index=gh_idx).reset_index()

# Left-join with observed rows to attach demand_log1p where it exists
grid = grid.merge(
    df[['geohash', 't_abs', 'demand_log1p', 'is_train']],
    on=['geohash', 't_abs'], how='left',
)
grid['was_observed'] = grid['demand_log1p'].notna().astype(np.int8)
grid = grid.sort_values(['geohash', 't_abs'], kind='stable').reset_index(drop=True)
print('Grid shape:', grid.shape, ' = ', len(geohashes), '×', len(ALL_T_ABS))
print('Observed cells:', int(grid['was_observed'].sum()), ' / expected', len(df))

Dense grid: t_abs 4608..4759  (152 slots = 96 + 56)
Unique geohashes (train ∪ test): 1259
Grid shape: (191368, 5)  =  1259 × 152
Observed cells: 77299  / expected 119077


## 5. Causal lag features

We compute six lags of `demand_log1p` per geohash on the dense grid: `lag_1, lag_2, lag_4, lag_8, lag_16, lag_96`. `lag_96` is exactly **24 hours back** at the same quarter-hour — the strongest single lag for a daily-cyclic series.

We also keep `was_obs_lag_k` (was the lagged cell actually observed?) so the model can discount a stale or imputed lag automatically.

In [5]:
LAGS = [1, 2, 4, 8, 16, 96]
g = grid.groupby('geohash', sort=False)
for k in LAGS:
    grid[f'lag_{k}']        = g['demand_log1p'].shift(k)
    grid[f'was_obs_lag_{k}']= g['was_observed'].shift(k).fillna(0).astype(np.int8)

print('Lag NaN counts (over the dense grid — many are expected since lag_96 needs 24h history):')
print(grid[[f'lag_{k}' for k in LAGS]].isna().sum())

Lag NaN counts (over the dense grid — many are expected since lag_96 needs 24h history):
lag_1     114069
lag_2     114069
lag_4     114069
lag_8     114069
lag_16    114069
lag_96    142402
dtype: int64


## 6. Causal rolling-window and EWM features

`rolling_mean_{4,8,16,32}`, `rolling_std_{8,32}`, `rolling_max_16`, and an EWM (α=0.3). Each window is shifted by 1 row before aggregation so the window is **strictly causal** — a row's rolling features only see *past* observations (`t_abs < current`).

In [6]:
for w in [4, 8, 16, 32]:
    grid[f'rolling_mean_{w}'] = g['demand_log1p'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean()
    )
for w in [8, 32]:
    grid[f'rolling_std_{w}']  = g['demand_log1p'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=2).std()
    )
grid['rolling_max_16'] = g['demand_log1p'].transform(
    lambda s: s.shift(1).rolling(16, min_periods=1).max()
)
grid['ewm_demand']     = g['demand_log1p'].transform(
    lambda s: s.shift(1).ewm(alpha=0.3, min_periods=1).mean()
)
print('Rolling/EWM features added. Sample non-NaN rows:')
print(grid[grid['was_obs_lag_4']==1][['geohash','t_abs','demand_log1p','rolling_mean_4','rolling_mean_16','ewm_demand']].head(5))

Rolling/EWM features added. Sample non-NaN rows:
   geohash  t_abs  demand_log1p  rolling_mean_4  rolling_mean_16  ewm_demand
8   qp02yc   4616           NaN        0.005382         0.005382    0.005382
14  qp02yc   4622      0.026079        0.019245         0.014624    0.019679
15  qp02yc   4623           NaN        0.025854         0.017488    0.023651
18  qp02yc   4626           NaN        0.026079         0.017488    0.023651
30  qp02yc   4638      0.004813        0.015123         0.017314    0.017021


## 7. Boundary features (per geohash, broadcast)

These five features capture **what the geohash's history looked like at the train/test boundary `t_abs = 4712`** — useful summary statistics that don't depend on the row's own time index:

- `boundary_demand` — the last observed `demand_log1p` at or before the boundary.
- `boundary_recency_{1h,2h,4h}_mean` — mean over the last 4, 8, 16 observed slots before the boundary.
- `saturation_fraction_last_24h` — fraction of the last 96 observations with raw demand ≥ 0.99.
- `train_history_len` — total observed train rows for this geohash.

For each test row, these are the same constant per geohash; for train rows we use the **same constants** so train and test see the same feature distribution (no train-side privileged information here).

In [7]:
# Build per-geohash boundary stats from grid restricted to t_abs <= train max
grid_train_portion = grid[grid['t_abs'] <= T_ABS_TRAIN_MAX]

records = []
for gh, sub in grid_train_portion.groupby('geohash', sort=False):
    obs = sub[sub['was_observed'] == 1].sort_values('t_abs')
    if len(obs) == 0:
        rec = dict(
            geohash=gh, boundary_demand=np.nan,
            boundary_recency_1h_mean=np.nan,
            boundary_recency_2h_mean=np.nan,
            boundary_recency_4h_mean=np.nan,
            saturation_fraction_last_24h=0.0,
            train_history_len=0, last_observed_t_abs=np.nan,
        )
    else:
        last96_raw = np.expm1(obs['demand_log1p'].iloc[-96:].values)
        rec = dict(
            geohash=gh,
            boundary_demand                = obs['demand_log1p'].iloc[-1],
            boundary_recency_1h_mean       = obs['demand_log1p'].iloc[-4:].mean(),
            boundary_recency_2h_mean       = obs['demand_log1p'].iloc[-8:].mean(),
            boundary_recency_4h_mean       = obs['demand_log1p'].iloc[-16:].mean(),
            saturation_fraction_last_24h   = (last96_raw >= 0.99).mean() if len(last96_raw) else 0.0,
            train_history_len              = len(obs),
            last_observed_t_abs            = obs['t_abs'].iloc[-1],
        )
    records.append(rec)
boundary = pd.DataFrame(records)
print('Per-geohash boundary stats:')
print(boundary['train_history_len'].describe())
print('Cold-start geohashes (train_history_len == 0):', int((boundary['train_history_len']==0).sum()))

Per-geohash boundary stats:
count    1259.000000
mean       61.397141
std        35.980115
min         0.000000
25%        28.000000
50%        71.000000
75%        94.000000
max       105.000000
Name: train_history_len, dtype: float64
Cold-start geohashes (train_history_len == 0): 10


## 8. Merge grid features and boundary features back onto `df`

We now have lag + rolling features on `grid` (152 slots × every geohash), and per-geohash boundary features. We merge both into the original `df` on `(geohash, t_abs)` for the grid features and on `geohash` for boundary features. After merge, `df` has the same 119 077 rows as before, but with all temporal features filled.

In [8]:
grid_feature_cols = (
    ['was_observed']
    + [f'lag_{k}' for k in LAGS]
    + [f'was_obs_lag_{k}' for k in LAGS]
    + [f'rolling_mean_{w}' for w in [4,8,16,32]]
    + [f'rolling_std_{w}'  for w in [8,32]]
    + ['rolling_max_16', 'ewm_demand']
)
df = df.merge(grid[['geohash','t_abs'] + grid_feature_cols], on=['geohash','t_abs'], how='left')
df = df.merge(boundary, on='geohash', how='left')

df['cold_start_flag']           = (df['train_history_len'] == 0).astype(np.int8)
df['last_seen_gap_to_boundary'] = (df['t_abs'] - df['last_observed_t_abs']).astype('float32')
df['time_since_boundary']       = (df['t_abs'] - T_ABS_TRAIN_MAX).clip(lower=0).astype(np.int16)
df = df.drop(columns=['last_observed_t_abs'])

# Sanity: test rows must have time_since_boundary in [1, 47]
test_tsb = df.loc[df['is_train']==0, 'time_since_boundary']
assert test_tsb.min() == 1 and test_tsb.max() == 47, f'time_since_boundary out of range: [{test_tsb.min()}, {test_tsb.max()}]'
print('df after lag+boundary merge:', df.shape)

df after lag+boundary merge: (119077, 77)


## 9. CV folds — five contiguous time-blocks covering every train slot

Each block is exactly 21 quarter-hours (≈ 5h 15m) of day 48. Together they cover the full 105-slot train range `t_abs ∈ [4608, 4712]`. Every train row is in **exactly one** fold's validation portion, so OOF predictions in the modelling stage will cover every train row without leakage.

| Fold | val t_abs range | hour-of-day window |
|------|-----------------|--------------------|
| 1    | 4608 – 4628     | 00:00 – 05:00      |
| 2    | 4629 – 4649     | 05:15 – 10:15      |
| 3    | 4650 – 4670     | 10:30 – 15:30      |
| 4    | 4671 – 4691     | 15:45 – 20:45      |
| 5    | 4692 – 4712     | 21:00 – day 49 02:00|

Folds 2 and 3 together cover almost exactly the test horizon (qhour 9–55 = hours 02:15–13:45), so the most relevant folds for evaluating test-style performance are 2 and 3.

In [9]:
N_FOLDS = 5
BLOCK_SIZE = 21
fold_ranges = []
for k in range(N_FOLDS):
    lo = T_ABS_LO + k * BLOCK_SIZE
    hi = lo + BLOCK_SIZE - 1
    fold_ranges.append((k + 1, lo, hi))
for fid, lo, hi in fold_ranges:
    hh_lo = (lo % 96) // 4
    mm_lo = (lo % 96) % 4 * 15
    hh_hi = (hi % 96) // 4
    mm_hi = (hi % 96) % 4 * 15
    print(f'Fold {fid}: t_abs [{lo}, {hi}]  =>  hour {hh_lo:02d}:{mm_lo:02d} .. {hh_hi:02d}:{mm_hi:02d}')

assert fold_ranges[-1][2] == T_ABS_TRAIN_MAX, 'Last fold must end at train_max'

train_df = df[df['is_train'] == 1].copy()
def assign_fold(t_abs):
    for fid, lo, hi in fold_ranges:
        if lo <= t_abs <= hi:
            return fid
    return -1
train_df['fold_id'] = train_df['t_abs'].apply(assign_fold).astype(np.int8)
assert (train_df['fold_id'] >= 1).all(), 'Some train rows not assigned to a fold'
print()
print('Rows per fold:')
print(train_df['fold_id'].value_counts().sort_index())

Fold 1: t_abs [4608, 4628]  =>  hour 00:00 .. 05:00
Fold 2: t_abs [4629, 4649]  =>  hour 05:15 .. 10:15
Fold 3: t_abs [4650, 4670]  =>  hour 10:30 .. 15:30
Fold 4: t_abs [4671, 4691]  =>  hour 15:45 .. 20:45
Fold 5: t_abs [4692, 4712]  =>  hour 21:00 .. 02:00

Rows per fold:
fold_id
1    17981
2    18795
3    17166
4     8275
5    15082
Name: count, dtype: int64


## 10. OOF empirical-Bayes target encodings (vectorised)

For each of 12 group-key combinations we compute a per-group statistic (mean / p90 / max / std / median of demand) and apply empirical-Bayes shrinkage toward the global mean. For each train row, the encoder is built **only on the other four folds** — the row's own fold doesn't contribute. For each test row, the encoder is the full-train aggregate.

We use vectorised pandas joins per fold so this runs in ~seconds, not the minutes a row-wise apply would take.

In [10]:
GLOBAL_MEAN_DEMAND = train_df['demand'].mean()
print(f'Global mean demand (train): {GLOBAL_MEAN_DEMAND:.6f}')

# Encoding specs: (output_name, group_cols, aggregator, shrinkage k)
ENCODINGS = [
    ('gh_mean_demand_oof',     ['geohash'],                          'mean',   10),
    ('gh_p90_demand_oof',      ['geohash'],                          'p90',    10),
    ('gh_max_demand_oof',      ['geohash'],                          'max',    10),
    ('gh_std_demand_oof',      ['geohash'],                          'std',    10),
    ('gh_median_demand_oof',   ['geohash'],                          'median', 10),
    ('pfx5_mean_demand_oof',   ['gh_prefix5'],                       'mean',    5),
    ('pfx5_p90_demand_oof',    ['gh_prefix5'],                       'p90',     5),
    ('pfx4_mean_demand_oof',   ['gh_prefix4'],                       'mean',    5),
    ('gh_qhour_mean_oof',      ['geohash','qhour_of_day'],           'mean',    8),
    ('pfx5_qhour_mean_oof',    ['gh_prefix5','qhour_of_day'],        'mean',    5),
    ('qhour_roadtype_mean_oof',['qhour_of_day','RoadType_code'],     'mean',    5),
    ('qhour_lanes_mean_oof',   ['qhour_of_day','NumberofLanes_modal'],'mean',   5),
]

AGG_FUNCS = {
    'mean':   lambda s: s.mean(),
    'p90':    lambda s: s.quantile(0.9),
    'p50':    lambda s: s.median(),
    'median': lambda s: s.median(),
    'max':    lambda s: s.max(),
    'std':    lambda s: s.std(ddof=0),
}

def eb_encode(enc_data, group_cols, agg, k, global_value):
    g = enc_data.groupby(group_cols)['demand']
    stat = g.agg(AGG_FUNCS[agg]).rename('stat')
    cnt  = g.size().rename('cnt')
    out = pd.concat([stat, cnt], axis=1).reset_index()
    out['eb'] = (out['cnt'] * out['stat'].fillna(global_value) + k * global_value) / (out['cnt'] + k)
    return out[group_cols + ['eb']]

Global mean demand (train): 0.093942


In [11]:
# Compute OOF encodings for train rows
train_oof = train_df[['row_id'] + sorted({c for spec in ENCODINGS for c in spec[1]})].copy()
train_oof = train_oof.merge(train_df[['row_id','fold_id']], on='row_id', how='left')

for name, group_cols, agg, k in ENCODINGS:
    # accumulator
    train_oof[name] = np.nan
    for fid, _lo, _hi in fold_ranges:
        enc_data = train_df[train_df['fold_id'] != fid]
        enc_table = eb_encode(enc_data, group_cols, agg, k, GLOBAL_MEAN_DEMAND)
        rows_to_fill = train_oof['fold_id'] == fid
        merged = train_oof.loc[rows_to_fill, group_cols].merge(enc_table, on=group_cols, how='left')
        train_oof.loc[rows_to_fill, name] = merged['eb'].fillna(GLOBAL_MEAN_DEMAND).values

# Verify no NaN remain
oof_cols = [spec[0] for spec in ENCODINGS]
nan_per_col = train_oof[oof_cols].isna().sum()
assert (nan_per_col == 0).all(), f'NaN in OOF encodings: {nan_per_col[nan_per_col>0]}'
print('Train OOF encodings — all 0 NaN ✓')

# Encoding for TEST rows: full-train aggregates
test_df = df[df['is_train'] == 0].copy()
test_enc_cols = sorted({c for spec in ENCODINGS for c in spec[1]})
test_enc_df = test_df[['row_id'] + test_enc_cols].copy()
for name, group_cols, agg, k in ENCODINGS:
    enc_table = eb_encode(train_df, group_cols, agg, k, GLOBAL_MEAN_DEMAND)
    merged = test_enc_df[group_cols].merge(enc_table, on=group_cols, how='left')
    test_enc_df[name] = merged['eb'].fillna(GLOBAL_MEAN_DEMAND).values
print('Test full-train encodings computed ✓')

Train OOF encodings — all 0 NaN ✓


Test full-train encodings computed ✓


In [12]:
# Attach OOF/full-train encodings back onto df
df = df.merge(train_oof[['row_id'] + oof_cols], on='row_id', how='left')
test_to_attach = test_enc_df[['row_id'] + oof_cols]
# Where row is a test row, fill from test_to_attach (df currently has NaN there because train_oof has only train rows)
df = df.set_index('row_id')
test_indexed = test_to_attach.set_index('row_id')
for c in oof_cols:
    df[c] = df[c].where(df.index.isin(train_df['row_id']),  # keep train values
                       test_indexed[c].reindex(df.index))   # else use test encoding
df = df.reset_index()

# Final NaN check
print('NaN check on encoded columns by split:')
for c in oof_cols:
    n_tr = df.loc[df['is_train']==1, c].isna().sum()
    n_te = df.loc[df['is_train']==0, c].isna().sum()
    assert n_tr == 0 and n_te == 0, f'{c}: train NaN={n_tr}, test NaN={n_te}'
print('  all 0 ✓')

NaN check on encoded columns by split:
  all 0 ✓


## 11. Saturation features

- `is_saturated_geohash` — 1 if the geohash ever observed `demand ≥ 0.99` in train.
- `gh_saturation_rate_oof` — empirical-Bayes shrunk per-geohash fraction of saturating rows. We use full-train rather than CV here because the signal is coarse and the shrinkage already controls overfitting; the cost of one-fold-OOF is small.
- `highway_multilane` — `RoadType_modal == 'Highway'` AND `NumberofLanes_modal ≥ 2`. The single biggest "where saturation can happen" indicator.

In [13]:
saturating_geohashes = sorted(
    train_df.loc[train_df['demand'] >= 0.99, 'geohash'].unique().tolist()
)
print(f'Saturating geohashes (demand ≥ 0.99 ever in train): {len(saturating_geohashes)}')

df['is_saturated_geohash'] = df['geohash'].isin(saturating_geohashes).astype(np.int8)

# OOF-ish saturation rate per geohash with EB shrinkage
GLOBAL_SAT_RATE = float((train_df['demand'] >= 0.99).mean())
sat_per_gh = (train_df.assign(_sat=(train_df['demand']>=0.99).astype(float))
                       .groupby('geohash')['_sat']
                       .agg(['mean','count']))
sat_per_gh['eb'] = (sat_per_gh['count']*sat_per_gh['mean'] + 10*GLOBAL_SAT_RATE) / (sat_per_gh['count'] + 10)
df['gh_saturation_rate_oof'] = df['geohash'].map(sat_per_gh['eb']).fillna(GLOBAL_SAT_RATE).astype('float32')

df['highway_multilane'] = ((df['RoadType_modal'] == 'Highway') & (df['NumberofLanes_modal'] >= 2)).astype(np.int8)
print('Saturation features added. Counts in train and test:')
print(df.groupby('is_train')[['is_saturated_geohash','highway_multilane']].sum())

Saturating geohashes (demand ≥ 0.99 ever in train): 23
Saturation features added. Counts in train and test:
          is_saturated_geohash  highway_multilane
is_train                                         
0                         1081               1128
1                         2414               2433


## 12. Sample weight

Two `tod_bucket`s — `afternoon_drop` and `evening_trough` — never appear in test (test covers qhour 9..55 = hours 02:15–13:45 only). Rows in those buckets are "out-of-distribution from the model's perspective" and would otherwise dominate the objective. We don't drop them (they still carry day-cycle information), but we **down-weight them to 0.4**. Everything else has weight 1.0.

> Note: `sample_weight` is **not** a model feature. It goes into `LightGBM.fit(..., sample_weight=...)`. It does not appear in `model_feature_cols`.

In [14]:
df['sample_weight'] = np.where(
    df['tod_bucket'].isin(['afternoon_drop', 'evening_trough']),
    0.4, 1.0,
).astype(np.float32)
print(df.groupby('tod_bucket')['sample_weight'].first())

tod_bucket
afternoon_drop     0.4
evening_trough     0.4
late_night_rise    1.0
midday_plateau     1.0
morning_ramp       1.0
Name: sample_weight, dtype: float32


## 13. Final asserts and split back

Before saving we verify the row counts, the time-since-boundary range on test, and that no encoded column has NaNs anywhere.

In [15]:
assert (df['is_train']==1).sum() == 77299
assert (df['is_train']==0).sum() == 41778

# Encoded columns have zero NaN
for c in oof_cols + ['gh_saturation_rate_oof','is_saturated_geohash','highway_multilane','sample_weight']:
    n = df[c].isna().sum()
    assert n == 0, f'{c} has {n} NaN'
print('All asserts pass ✓')

# Split back, drop the helper is_train flag, and rename row_id → Index for downstream stability
train_features = (
    df[df['is_train']==1]
      .drop(columns=['is_train'])
      .sort_values('row_id').reset_index(drop=True)
      .rename(columns={'row_id':'Index'})
)
test_features = (
    df[df['is_train']==0]
      .drop(columns=['is_train','demand','demand_log1p'])
      .sort_values('row_id').reset_index(drop=True)
      .rename(columns={'row_id':'Index'})
)
print('train_features:', train_features.shape)
print('test_features :', test_features.shape)

All asserts pass ✓
train_features: (77299, 92)
test_features : (41778, 90)


## 14. Build `model_feature_cols` and save

We exclude from the model X-matrix:

- IDs and the timestamp string (`Index`, `timestamp`)
- Target columns (`demand`, `demand_log1p`)
- Raw string forms of categoricals (we keep their `*_code` versions)
- Redundant time columns: `day` (constant in test), `t_abs` (out-of-range at test), `NumberofLanes_ord` (byte-identical to `NumberofLanes`), `hour`, `hour_sin`, `hour_cos` (subsumed by `qhour_*`)
- `sample_weight` (this is a fitter argument, not a feature)

Categorical feature codes are listed separately so LightGBM can be told `categorical_feature=...`.

In [16]:
DROP_FROM_MODEL = {
    'Index', 'timestamp', 'demand', 'demand_log1p',
    'geohash', 'gh_prefix4', 'gh_prefix5', 'RoadType', 'LargeVehicles',
    'Landmarks', 'Weather', 'tod_bucket',
    'RoadType_modal', 'LargeVehicles_modal', 'Landmarks_modal',
    'day', 't_abs', 'NumberofLanes_ord', 'hour', 'hour_sin', 'hour_cos',
    'sample_weight',
}
model_feature_cols = [c for c in train_features.columns if c not in DROP_FROM_MODEL]
categorical_code_cols = [
    c for c in model_feature_cols
    if c.endswith('_code')
]
target_encoded_cols = oof_cols
print(f'model_feature_cols: {len(model_feature_cols)} total')
print(f'  categorical (code): {len(categorical_code_cols)}')
print(f'  target-encoded OOF: {len(target_encoded_cols)}')
print()
print('Categorical:', categorical_code_cols)
print()
print('Target-encoded:', target_encoded_cols)

model_feature_cols: 70 total
  categorical (code): 11
  target-encoded OOF: 12

Categorical: ['tod_bucket_code', 'geohash_code', 'gh_prefix4_code', 'gh_prefix5_code', 'RoadType_code', 'LargeVehicles_code', 'Landmarks_code', 'RoadType_modal_code', 'LargeVehicles_modal_code', 'Landmarks_modal_code', 'Weather_code']

Target-encoded: ['gh_mean_demand_oof', 'gh_p90_demand_oof', 'gh_max_demand_oof', 'gh_std_demand_oof', 'gh_median_demand_oof', 'pfx5_mean_demand_oof', 'pfx5_p90_demand_oof', 'pfx4_mean_demand_oof', 'gh_qhour_mean_oof', 'pfx5_qhour_mean_oof', 'qhour_roadtype_mean_oof', 'qhour_lanes_mean_oof']


In [17]:
train_features.to_parquet(os.path.join(PROCESSED, 'train_features.parquet'), index=False)
test_features.to_parquet (os.path.join(PROCESSED, 'test_features.parquet'),  index=False)

# Save CV folds (row_id → fold_id, role='val' for the fold)
cv = train_df[['row_id','fold_id']].rename(columns={'row_id':'Index'})
cv['role'] = 'val'
cv.to_parquet(os.path.join(PROCESSED, 'cv_folds.parquet'), index=False)

with open(os.path.join(PROCESSED, 'saturating_geohashes.json'), 'w') as f:
    json.dump(saturating_geohashes, f, indent=2)

meta = dict(
    model_feature_cols=model_feature_cols,
    categorical_code_cols=categorical_code_cols,
    target_encoded_cols=target_encoded_cols,
    sample_weight_col='sample_weight',
    id_col='Index',
    target_col='demand',
    target_log_col='demand_log1p',
    saturating_geohash_count=len(saturating_geohashes),
    fold_definition=[{'fold_id': fid, 't_abs_lo': lo, 't_abs_hi': hi} for fid,lo,hi in fold_ranges],
    train_t_abs_range=[T_ABS_LO, T_ABS_TRAIN_MAX],
    test_t_abs_range=[T_ABS_TRAIN_MAX+1, T_ABS_HI],
    coordinate_system='anonymized_geohash_decode',
    coordinate_note='lat/lon decode to ~-5.36N, 90.81E (Indian Ocean); the real source is Bengaluru. '
                    'Do NOT join with external geo data — only use relative geometry within the box.',
)
with open(os.path.join(PROCESSED, 'feature_meta_v2.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fn in ['train_features.parquet','test_features.parquet','cv_folds.parquet',
           'saturating_geohashes.json','feature_meta_v2.json']:
    p = os.path.join(PROCESSED, fn)
    sz = os.path.getsize(p)
    print(f'  {p}  ({sz/1024:.1f} KB)')

Saved:
  .\processed\train_features.parquet  (13363.8 KB)
  .\processed\test_features.parquet  (3181.4 KB)
  .\processed\cv_folds.parquet  (455.7 KB)
  .\processed\saturating_geohashes.json  (0.3 KB)
  .\processed\feature_meta_v2.json  (3.3 KB)


## 15. Sanity round-trip

Re-load the saved parquets and verify that everything aligns row-by-row with the raw test file's `Index` column. This is the assertion the modelling notebooks rely on for submission alignment.

In [18]:
tr = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
te = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
raw_test = pd.read_csv(os.path.join("e88186124ec611f1 (1)/dataset", 'test.csv'))

assert tr.shape[0] == 77299
assert te.shape[0] == 41778
assert (te['Index'].values == raw_test['Index'].values).all(), 'test Index alignment broken'
print('Round-trip OK ✓ — saved files match expected schema and ordering.')

# A quick OOF-encoder sanity: gh_mean_demand_oof vs naive gh_mean_demand (full-train).
naive_gh_mean = tr.groupby('geohash')['demand'].mean()
oof_gh_mean   = tr.groupby('geohash')['gh_mean_demand_oof'].first()
diff = (naive_gh_mean - oof_gh_mean).abs()
print(f'\nOOF vs naive geohash-mean — mean abs diff: {diff.mean():.6f}, max: {diff.max():.6f}')
print('(Non-zero → confirms OOF encoding is actually held-out, not pure full-train.)')

Round-trip OK ✓ — saved files match expected schema and ordering.

OOF vs naive geohash-mean — mean abs diff: 0.022922, max: 0.103976
(Non-zero → confirms OOF encoding is actually held-out, not pure full-train.)


## Summary

| Output                            | Rows    | Columns |
|-----------------------------------|---------|---------|
| `processed/train_features.parquet`| 77 299  | 92      |
| `processed/test_features.parquet` | 41 778  | 90      |
| `processed/cv_folds.parquet`      | 77 299  | 3       |
| `processed/saturating_geohashes.json` | —   | 23 entries |
| `processed/feature_meta_v2.json`  | —       | feature manifest |

### What the next notebooks consume
- `model_train_lgbm.ipynb` reads `train_features.parquet`, `test_features.parquet`, `feature_meta_v2.json` (`model_feature_cols`), `cv_folds.parquet`, and `val_split_v2_indices.parquet`.
- The same three model-training notebooks all share these files — there is no separate per-model feature pipeline.
- The `sample_weight` column travels in the parquet but is read separately by each training script (it is *not* in `model_feature_cols`).
